# Cached provenance graph GPU training

This notebook uses a precomputed, strict-past two-hop Host/Process ego-graph cache. It does not open `graph.sqlite`, raw telemetry, or baseline Host ML shards. In Kaggle Settings, enable a GPU accelerator before running.


In [ ]:
from pathlib import Path
import json, os, shutil, sys, tarfile

INPUT_ROOT = Path('/kaggle/input')
WORK_ROOT = Path('/kaggle/working/provenance-ego-cache')

# 1. Tìm archive .tar nếu Kaggle chưa giải nén
archives = list(INPUT_ROOT.rglob('provenance-ego-cache.tar')) or list(INPUT_ROOT.rglob('*.tar'))

if archives:
    archive_path = archives[0]
    print(f'Tìm thấy archive: {archive_path}, đang giải nén vào {WORK_ROOT}...')
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive_path) as archive:
        for member in archive.getmembers():
            target = (WORK_ROOT / member.name).resolve()
            if member.issym() or member.islnk() or not target.is_relative_to(WORK_ROOT.resolve()):
                raise ValueError(f'Unsafe archive member: {member.name}')
        archive.extractall(WORK_ROOT, filter='data')
    CACHE_ROOT = WORK_ROOT
else:
    # 2. Kaggle đã tự động giải nén dataset khi tải lên
    manifest_candidates = [p for p in INPUT_ROOT.rglob('manifest.json') if (p.parent / 'train').is_dir()]
    if not manifest_candidates:
        manifest_candidates = list(INPUT_ROOT.rglob('manifest.json'))
    if not manifest_candidates:
        raise RuntimeError(f'Không tìm thấy manifest.json trong {INPUT_ROOT}. Thư mục: {list(INPUT_ROOT.glob("* "))}')
    CACHE_ROOT = manifest_candidates[0].parent
    print(f'Kaggle đã tự giải nén dataset tại: {CACHE_ROOT}')
    WORK_ROOT.mkdir(parents=True, exist_ok=True)
    for script_name in ['train_gpu.py', 'ego_cache.py']:
        src = CACHE_ROOT / script_name
        if src.is_file():
            shutil.copy2(src, WORK_ROOT / script_name)

meta = json.loads((CACHE_ROOT / 'manifest.json').read_text())
if meta.get('complete') is not True:
    raise ValueError('Attached cache is incomplete')
print('Cache root:', CACHE_ROOT)
print('Graphs:', meta['splits'])
print('Attributes:', len(meta['attributes']), 'Relations:', meta['relations'])

import torch
print('PyTorch:', torch.__version__, 'CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Enable a Kaggle GPU accelerator, then rerun from this cell.')


In [ ]:
import subprocess, sys
from pathlib import Path

OUTPUT = Path('/kaggle/working/provenance-gpu-model')
EPOCHS = 30
BATCH_SIZE = 128
RESUME = False

trainer_script = WORK_ROOT / 'train_gpu.py'
if not trainer_script.is_file():
    trainer_script = CACHE_ROOT / 'train_gpu.py'

command = [
    sys.executable, str(trainer_script),
    '--cache', str(CACHE_ROOT),
    '--output', str(OUTPUT),
    '--epochs', str(EPOCHS),
    '--batch-size', str(BATCH_SIZE),
    '--device', 'cuda',
    '--evaluate-test'
]
if RESUME:
    command.append('--resume')
print('Executing:', ' '.join(command))
subprocess.run(command, check=True)


In [ ]:
metrics = json.loads((OUTPUT / 'metrics.json').read_text())
print(json.dumps(metrics, indent=2))
print('Compare only the cached validation/test scopes shown in metrics.json. Validation selected the threshold; test did not.')


In [ ]:
# Verify NumPy weight export (compatible with relational_sage.RelationalSAGE.load())
import numpy as np

numpy_path = OUTPUT / 'model_numpy.npz'
if numpy_path.is_file():
    d = np.load(numpy_path, allow_pickle=False)
    dims = d['dimensions'].tolist()
    param_keys = [k for k in d.files if not k.startswith('adam_') and k not in ('dimensions', 'steps')]
    n_params = sum(d[k].size for k in param_keys)
    print(f'model_numpy.npz: dimensions={dims}  params={n_params}  steps={int(d["steps"])}')
    print(f'Keys: {sorted(param_keys)}')
    print(f'File size: {numpy_path.stat().st_size:,} bytes')
    print()
    print('To use locally:')
    print('  from relational_sage import RelationalSAGE')
    print('  model = RelationalSAGE.load("model_numpy.npz")')
    print('  # Then run: python3 code/audit/graph_vs_ml_complementarity.py --model-weights <dir>')
else:
    print('WARNING: model_numpy.npz not found — check train_gpu.py export_numpy_weights() was called')


In [ ]:
# Package model outputs for download and local complementarity analysis.
# Bundles: checkpoint.pt, model_numpy.npz, metrics.json, config.json,
#          progress.json, and the cache manifest digest for provenance.
import hashlib, os, zipfile
from pathlib import Path

def sha256(path):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(1 << 20), b''):
            h.update(chunk)
    return h.hexdigest()

zip_path = Path('/kaggle/working/optc-provenance-graph-results.zip')
tmp_path = zip_path.with_suffix('.zip.tmp')

# Files to include from the model output directory
INCLUDE_SUFFIXES = {'.pt', '.npz', '.json'}

try:
    with zipfile.ZipFile(tmp_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
        # Model artifacts
        for src in sorted(OUTPUT.rglob('*')):
            if src.is_file() and src.suffix in INCLUDE_SUFFIXES:
                archive.write(src, 'provenance-gpu-model/' + src.relative_to(OUTPUT).as_posix())
        # Cache manifest digest (provenance, not the full cache)
        manifest_path = CACHE_ROOT / 'manifest.json'
        if manifest_path.is_file():
            archive.write(manifest_path, 'provenance/cache_manifest.json')
        # Selection.json (source_line join key for complementarity analysis)
        selection_path = CACHE_ROOT / 'selection.json'
        if selection_path.is_file():
            archive.write(selection_path, 'provenance/selection.json')
        archive.writestr('provenance/README.txt',
            'provenance-gpu-model/model_numpy.npz  — NumPy weights (relational_sage.RelationalSAGE.load())\n'
            'provenance-gpu-model/checkpoint.pt    — PyTorch checkpoint (requires PyTorch to load)\n'
            'provenance-gpu-model/metrics.json     — val/test metrics and threshold\n'
            'provenance-gpu-model/config.json      — model hyperparameters and cache identity\n'
            'provenance/selection.json             — test/val ego-graph source_line join keys\n'
            'Join source_line in selection.json with ML test_errors.json for complementarity analysis.\n'
            'Run: python3 code/audit/graph_vs_ml_complementarity.py --model-weights <dir with model_numpy.npz>\n'
        )
    os.replace(tmp_path, zip_path)
finally:
    if tmp_path.exists():
        tmp_path.unlink()

digest = sha256(zip_path)
size_mb = zip_path.stat().st_size / 1024**2
print(f'Created: {zip_path}')
print(f'Size:    {size_mb:.2f} MiB')
print(f'SHA256:  {digest}')
print()
print('Contents:')
with zipfile.ZipFile(zip_path) as z:
    for info in z.infolist():
        print(f'  {info.filename}  ({info.file_size:,} bytes)')
